# Periodic 3D Voronoi grain structure in UPXO
### *Generates a periodic Voronoi grain structure on a voxel grid of 50 x 75 x 100 voxels.*

In a periodic structure the faces of the box wrap around. A grain that leaves the box through the right face comes back in through the left face, and it carries the same grain ID on both sides. The same holds for the front and back faces and for the bottom and top faces.

You will need a UPXO release that includes `generate_voronoi_lfi_periodic` (added after 1.5.1). Install with `pip install upxo`. The 3D view in section 5 also needs PyVista (`pip install upxo[all]` installs it).

This notebook only generates the structure and inspects it. The generator is `upxo.pxtal.voronoi_tessellation_3d.generate_voronoi_lfi_periodic`.

**What periodic means here.** The grain IDs are periodic. A grain that wraps across a face is stored as two separate blobs of voxels with one ID. Anything that finds grain neighbours from voxels touching inside the array does not link grains across the seam.

## Setup

Import statements first. *Most users need not worry about this*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from upxo.pxtal.voronoi_tessellation_3d import generate_voronoi_lfi_periodic

## 1. Generate the periodic Voronoi structure

Random seed points are placed uniformly in the box. Each seed becomes one grain, so `N_SEEDS` is the number of grains you get (a grain too small to contain a voxel centre is lost, which is rare at this resolution). `VORONOI_SEED` makes the placement repeatable.

Each voxel centre is given to its nearest seed, measured the periodic way: the distance to a seed is taken across the nearest copy of the box, whichever side of the box that copy is on.

The grid is `NX x NY x NZ` voxels and each voxel is `VOXEL_SIZE` long, so the physical size of the box is the number of voxels times `VOXEL_SIZE`. The result `lfi` is an integer array of shape `(NX, NY, NZ)` holding the grain ID, numbered from 1, of every voxel.

In [ ]:
NX, NY, NZ = 50, 75, 100     # voxels along x, y, z
VOXEL_SIZE = 1.0             # edge length of one voxel
UNITS = 'microns'            # unit of VOXEL_SIZE
N_SEEDS = 100                # number of Voronoi cells (grains)
VORONOI_SEED = 0             # seed for the random point placement

LX, LY, LZ = NX * VOXEL_SIZE, NY * VOXEL_SIZE, NZ * VOXEL_SIZE

lfi = generate_voronoi_lfi_periodic(NX, NY, NZ, N_SEEDS, VORONOI_SEED, lx=LX, ly=LY, lz=LZ)
print(f'{len(np.unique(lfi))} grains on a {NX} x {NY} x {NZ} grid '
      f'({LX:g} x {LY:g} x {LZ:g} {UNITS})')

## 2. Grain sizes

Number of voxels in each grain, and the volume that corresponds to.

In [ ]:
n_voxels = np.bincount(lfi.ravel())[1:]          # voxels per grain ID 1..N
volume = n_voxels * VOXEL_SIZE ** 3

print(f'total voxels  : {lfi.size}')
print(f'voxels / grain: min {n_voxels.min()}, mean {n_voxels.mean():.0f}, max {n_voxels.max()}')
print(f'volume / grain: min {volume.min():g}, mean {volume.mean():g}, max {volume.max():g} {UNITS}^3')

## 3. Slices

One slice through the middle of the box normal to each axis. Each grain has its own random colour. The colours are only labels.

In [ ]:
def grain_cmap(n_ids, seed=1):
    rng = np.random.default_rng(seed)
    colours = rng.random((n_ids + 1, 3))
    colours[0] = 1.0
    return ListedColormap(colours)

cmap = grain_cmap(int(lfi.max()))
vmax = int(lfi.max())

slices = [
    ('x-y plane (z = mid)', lfi[:, :, NZ // 2].T, (0, LX, 0, LY), 'x', 'y'),
    ('x-z plane (y = mid)', lfi[:, NY // 2, :].T, (0, LX, 0, LZ), 'x', 'z'),
    ('y-z plane (x = mid)', lfi[NX // 2, :, :].T, (0, LY, 0, LZ), 'y', 'z'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img, extent, xl, yl) in zip(axes, slices):
    ax.imshow(img, origin='lower', extent=extent, cmap=cmap, vmin=0, vmax=vmax,
              interpolation='nearest')
    ax.set_title(title)
    ax.set_xlabel(f'{xl} ({UNITS})')
    ax.set_ylabel(f'{yl} ({UNITS})')
plt.tight_layout()
plt.show()

## 4. Check that the structure is periodic

**Across the seam.** For two neighbouring layers of voxels, the share of voxel pairs that carry the same grain ID is high, because most voxels have the same grain as the voxel next to them. Taking the last layer and the first layer of an axis as neighbours (the seam) should give about the same share as two neighbouring layers inside the box. If the faces were unrelated, the seam share would be close to zero.

**Tiled view.** The middle x-y slice is repeated 2 x 2. Where the tiles meet, the grains continue across the dashed lines.

In [ ]:
print('axis   inside the box   across the seam')
for axis, name in enumerate('xyz'):
    n = lfi.shape[axis]
    inside = np.mean([np.mean(np.take(lfi, i, axis=axis) == np.take(lfi, i + 1, axis=axis))
                      for i in range(n - 1)])
    seam = np.mean(np.take(lfi, n - 1, axis=axis) == np.take(lfi, 0, axis=axis))
    print(f'  {name}        {inside:.3f}            {seam:.3f}')

tile = np.tile(lfi[:, :, NZ // 2].T, (2, 2))
fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(tile, origin='lower', extent=(0, 2 * LX, 0, 2 * LY), cmap=cmap, vmin=0, vmax=vmax,
          interpolation='nearest')
ax.axvline(LX, color='k', ls='--', lw=1)
ax.axhline(LY, color='k', ls='--', lw=1)
ax.set_title('x-y slice repeated 2 x 2')
ax.set_xlabel(f'x ({UNITS})')
ax.set_ylabel(f'y ({UNITS})')
plt.show()

## 5. 3D view

The voxel grid is shown as a PyVista image grid coloured by grain ID. `VIZ_BACKEND = 'trame'` shows it inside the notebook so you can rotate it. Use `None` to open it as a separate interactive view instead. The grain IDs are shuffled before colouring, so neighbouring grains don't end up with similar colours.

In [ ]:
import pyvista as pv

VIZ_BACKEND = 'trame'            # 'trame' for an in-notebook view, None for a separate view

relabel = np.concatenate([[0], np.random.default_rng(0).permutation(np.arange(1, vmax + 1))])
shuffled_ids = relabel[lfi]
grid = pv.ImageData(dimensions=(NX + 1, NY + 1, NZ + 1),
                    spacing=(VOXEL_SIZE,) * 3, origin=(0.0, 0.0, 0.0))
grid.cell_data['grain'] = shuffled_ids.ravel(order='F')
grid.plot(scalars='grain', cmap=cmap, clim=(0, vmax), show_scalar_bar=False,
          jupyter_backend=VIZ_BACKEND)